In [81]:
import numpy as np
from dataclasses import dataclass
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt

class Params:
    # Geometry / discretization
    
    L_fiber: float = 50
    n_fibers: int = 4
    dx: float = 0.1
    n_rows: int = 5
    row_width: int = 4 #mm
    resin_gap: float =  0.1 #mm
    L: float = (L_fiber+resin_gap)*n_fibers #mm
    # Loading
    eps0: float = 0.01  # applied macroscopic strain (displacement control)

    # Tow properties (axial)
    E_tow: float = 140e9
    D_f = 4
    t_f = .1
    A_tow: float = D_f * t_f # cross-sectional area of a single tow

    row_break = [L*.49, L*.51]
    # Matrix/interface (shear transfer)
    G_m: float = 1.0e9
    E_m: float = 3.0e9
    t_eff: float = .10 #mm
    b_eff: float = 1.0 #mm  # effective width for patch area
#======================================================================================
#staggering helper

def staggering_coords(params: Params):
    row_breakseven = []
    row_breaksodd = []

    for i in range(params.n_fibers):
        # even row breaks
        x_even = round(i * params.L_fiber + (i - 1) * params.resin_gap, 2)
        y_even = round(i * (params.L_fiber + params.resin_gap), 2)
        row_breakseven.append((x_even, y_even))

        # odd row breaks
        x_odd = round((i + 1) * params.L_fiber + i * params.resin_gap - params.L_fiber / 2, 2)
        y_odd = round((i + 1) * (params.L_fiber + params.resin_gap) - params.L_fiber / 2, 2)
        row_breaksodd.append((x_odd, y_odd))

    # remove the first element from even list if desired
    row_breakseven = row_breakseven[1:]

    return row_breakseven, row_breaksodd
print(staggering_coords(Params()))

def staggering_coords(params: Params):
    row_breakseven = []
    row_breaksodd = []

    for i in range(params.n_fibers):
        # even row breaks
        x_even = round(i * params.L_fiber + (i - 1) * params.resin_gap, 2)
        y_even = round(i * (params.L_fiber + params.resin_gap), 2)
        row_breakseven.append((x_even, y_even))

        # odd row breaks
        x_odd = round((i + 1) * params.L_fiber + i * params.resin_gap - params.L_fiber / 2, 2)
        y_odd = round((i + 1) * (params.L_fiber + params.resin_gap) - params.L_fiber / 2, 2)
        row_breaksodd.append((x_odd, y_odd))

    # remove the first element from even list if desired
    row_breakseven = row_breakseven[1:]

    return row_breakseven, row_breaksodd


#======================================================================================
# indexing help
#=======================================================================================
def build_mesh(L: float, dx: float) -> np.ndarray:
    # Ensure endpoint included
    n = int(np.round(L / dx))
    #x = np.linspace(0.0, L, n + 1)
    x=  np.arange(n+1, dtype=float) * dx
    x[-1] = L
    return x.round(2)


def dof_index(row: int, node: int, n_nodes: int) -> int:
    # global DOF index for 1 DOF per node per row
    return round(row * n_nodes + node,3)
#======================================================================================
#element assembly
#=====================================================================================


print( build_mesh(Params.L, Params.dx)  )

def add_2node_spring(K, a: int, b: int, k: float)-> None:
    K[a, a] += k
    K[b, b] += k
    K[a, b] -= k
    K[b, a] -= k


def k_axial(E_tow: float, A_tow: float, dx: float) -> float:
    return (E_tow * A_tow) / dx

def k_shear(G_m: float, b_eff: float, t_eff: float, dx: float) -> float:
    # shear spring stiffness per unit length
    k = G_m * b_eff * dx / t_eff
    return k

#======================================================================================
#boundary conditions
#======================================================================================
def apply_dirichlet_elimination(K, f, prescribed: dict[int, float]):
    """
    Dirichlet elimination:
      - prescribed: {dof_index: value}
    Returns:
      - K_ff, f_eff, free_dofs, u_full (with prescribed entries filled)
    """
    n = f.shape[0] # we dont actually import force displacemnts just there in form ku = f 
    u_full = np.zeros(n)


    # fill prescribed dof in u_full
    for dof_index, value in prescribed.items():
        u_full[dof_index] = value

    prescribed_dofs = np.array(sorted(prescribed.keys())) #converts dof indices and sorts them in array
    is_prescribed = np.zeros(n, dtype=bool) # creates boolean array of size n
    is_prescribed[prescribed_dofs] = True # where no dof is prescribed, set to True
    free_dofs = np.where(~is_prescribed)[0] #flips booleans and sptis out indicies of free dofs
    # get submatricies k_ff = k[free, free] kfp = K free, prescribed
    K_ff = K[free_dofs][:,free_dofs] #select row and col corresponding to free dofs
    K_gp = K[free_dofs][:, prescribed_dofs] # select row corresponding to free dofs and col corresponding to prescribed dofs


    #rhs f_f = K_fp*u+p
    u_p = u_full[prescribed_dofs] # get prescribed displacements
    f_f = f[free_dofs] # global forces corresponding with global free dofs #typicall 0 
    f_eff = f_f - K_gp @ u_p # effective rhs # equivalent load resulting from displacement
    return K_ff, f_eff, free_dofs, u_full



([(50.0, 50.1), (100.1, 100.2), (150.2, 150.3)], [(25.0, 25.1), (75.1, 75.2), (125.2, 125.3), (175.3, 175.4)])
[0.000e+00 1.000e-01 2.000e-01 ... 2.002e+02 2.003e+02 2.004e+02]


In [ ]:
def assemble_phase1(params: Params, x:np.ndarray):
    n_nodes = len(x)
    n_dofs = params.n_rows * n_nodes


    #list of list sparse matrix
    K = lil_matrix((n_dofs, n_dofs))
    f = np.zeros(n_dofs, dtype= float)


    #axial springs in both rows (continuous to start) 
    kax = k_axial(params.E_tow, params.A_tow, params.dx)
    ksh = k_shear(params.G_m, params.b_eff, params.t_eff, params.dx)
    kbridge = params.E_m*params.A_tow/(params.dx) # spring constant for shear spring in break region
    elems = [] 
    evenoffset, oddoffset = staggering_coords(params)
    for row in range(params.n_rows):
        if row %2 ==0:

            for i in range(n_nodes-1):
                 xmid = (x[i]+x[i+1])/2 
                 inside = any(xstart<=xmid<=xend for (xstart, xend) in evenoffset)
                 if not inside:
                          a = dof_index(row, i, n_nodes)
                          b = dof_index(row, i+1, n_nodes)
                          add_2node_spring(K, a, b, kax)
                          elems.append({"etype": "tow_axial", "row": row, "i": i, "a": a, "b": b, "k": kax, "A": params.A_tow})
                 if inside: 
                        a = dof_index(row, i, n_nodes)
                        b = dof_index(row, i+1, n_nodes)
                        add_2node_spring(K, a, b, kbridge)
                        elems.append({"etype": "tow_bridge", "row": row, "i": i, "a": a, "b": b, "k": kbridge, "A": params.A_tow})
                 
        if row %2 !=0:
            for i in range(n_nodes-1):
                xmid = (x[i]+x[i+1])/2
                inside = any(xstart <= xmid <= xend for (xstart, xend) in oddoffset)
                if not inside: 
                    a = dof_index(row, i, n_nodes)
                    b = dof_index(row, i+1, n_nodes)
                    add_2node_spring(K, a, b, kax)
                    elems.append({"etype": "tow_axial", "row": row, "i": i, "a": a, "b": b, "k": kax, "A": params.A_tow})
                if inside:
                    a = dof_index(row, i, n_nodes)
                    b = dof_index(row, i+1, n_nodes)
                    add_2node_spring(K, a, b, kbridge)
                    elems.append({"etype": "tow_bridge", "row": row, "i": i, "a": a, "b": b, "k": kbridge, "A": params.A_tow})


      
    for r in range(params.n_rows - 1):
        for i, xi in enumerate(x):
                a = dof_index(r, i, n_nodes)
                b = dof_index(r+1, i, n_nodes)
                add_2node_spring(K, a, b, ksh)
                elems.append({"etype": "shear", "i": i, "a": a, "b": b,  "k": ksh})

    return K.tocsr(), f, elems # global stiffness matrix and global applied force vector correspondign for ku=f

[0.000e+00 1.000e-01 2.000e-01 ... 2.002e+02 2.003e+02 2.004e+02]
13


In [ ]:
import numpy as np
class Params:
    # Geometry / discretization
    
    L_fiber: float = 50e-3
    n_fibers: int = 4
    dx: float = 0.1e-3
    n_rows: int = 5
    row_width: int = 4e-3 #mm
    resin_gap: float =  0.5e-3 #mm
    L: float = (L_fiber+resin_gap)*n_fibers #mm
    # Loading
    eps0: float = 0.01  # applied macroscopic strain (displacement control)

    # Tow properties (axial)
    E_tow: float = 140e9
    D_f = 4e-3
    t_f = .1e-3
    A_tow: float = D_f * t_f # cross-sectional area of a single tow

    row_break = [L*.49, L*.51]
    # Matrix/interface (shear transfer)
    G_m: float = 1.0e9
    E_m: float = 3.0e9
    t_eff: float = .10e-3 #mm
    b_eff: float = 1.0e-3 #mm  # effective width for patch area
#======================================================================================
#staggering helper
def staggering_coords(params: Params, odd_shift: float = None):
    """
    Return x-intervals (start,end) for resin gaps for:
      - even rows: no shift
      - odd rows: shifted by odd_shift (default = L_fiber/2)

    All intervals are clipped to [0, params.L].
    """
    Lf = float(params.L_fiber)
    g  = float(params.resin_gap)
    L  = float(params.L)

    if odd_shift is None:
        odd_shift = 0.5 * Lf  # classic half-brick staggering

    pitch = Lf + g

    def clip_interval(a, b):
        a2 = max(0.0, a)
        b2 = min(L, b)
        if b2 <= a2:
            return None
        return (round(a2, 6), round(b2, 6))

    # ---- Even-row gaps (no shift) ----
    even_gaps = []
    # gap k sits right after tow segment k: [end_of_tow_k, end_of_tow_k + g]
    # end_of_tow_k = (k+1)*Lf + k*g
    # choose k range large enough to cover domain
    k_max = int(np.ceil(L / pitch)) + 2

    for k in range(k_max):
        x0 = (k + 1) * Lf + k * g
        x1 = x0 + g
        iv = clip_interval(x0, x1)
        if iv is not None:
            even_gaps.append(iv)

    # ---- Odd-row gaps (shifted) ----
    odd_gaps = []
    # shift the entire pattern left by odd_shift (equivalently: subtract odd_shift)
    for k in range(k_max):
        x0 = (k + 1) * Lf + k * g - odd_shift
        x1 = x0 + g
        iv = clip_interval(x0, x1)
        if iv is not None:
            odd_gaps.append(iv)

    return even_gaps, odd_gaps
print(staggering_coords(Params()))

([(0.05, 0.0505), (0.1005, 0.101), (0.151, 0.1515), (0.2015, 0.202)], [])
